# RetinaSafe — Step 8b: DINOv2 fine-tuning v2 (lower LR)

**Why v2:** Step 8 used LR=1e-4 with 2-epoch warmup. DINOv2's self-distillation-pretrained features are sensitive to aggressive fine-tuning, and that recipe collapsed the model to macro AUROC 0.7293 — worse than the linear probe at 0.8965.

**Two changes only:**
1. **Learning rate: 1e-5** (10× lower than v1)
2. **Warmup epochs: 5** (instead of 2) — longer warmup gives DINOv2 time to settle into the new task before LR ramps up

Everything else (model, data, splits, AMP, BCE+pos_weight, max 20 epochs, patience 6, eval pipeline) is identical to Step 8 v1.

**Inputs:**
1. `samarthmishra208/brset-brazilian-multilabel-ophthalmological`
2. **Notebook Output:** `samarthmishra208/brset-baseline` (for `brset_index.csv`)

**Settings:** GPU T4 x2, **Internet ON** (torch.hub downloads DINOv2 if not cached). Runtime ~3 hr.

**Three possible outcomes:**
- **≥ 0.90 macro AUROC** → recipe was the issue; DINOv2 just needs gentler FT. Most expected.
- **0.85–0.89** → DINOv2 doesn't benefit much from FT on this dataset. Probe is the right way to use it.
- **Still < 0.85** → real finding: DINOv2 truly is best used frozen on BRSET.

## Cell 1 — Imports + paths

In [ ]:
import os, sys, json, time, random, glob, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

img_dirs = glob.glob("/kaggle/input/**/fundus_photos", recursive=True)
assert img_dirs, "BRSET fundus_photos not found"
IMAGES_DIR = img_dirs[0]
idx_csvs = glob.glob("/kaggle/input/**/brset_index.csv", recursive=True)
assert idx_csvs, "brset_index.csv not found"
INDEX_CSV = idx_csvs[0]
print(f"\nIMAGES_DIR: {IMAGES_DIR}")
print(f"INDEX_CSV: {INDEX_CSV}")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)
CKPT = RESULTS / "best.pt"

DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]
N_LABELS = len(DISEASE_COLS)

## Cell 2 — Dataset + transforms (same as RETFound FT)

In [ ]:
IMG_SIZE = 224  # multiple of 14, works for DINOv2 ViT-L/14
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def train_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size + 32, size + 32)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class BRSETDataset(Dataset):
    def __init__(self, index_csv, split, images_dir, transform=None):
        full = pd.read_csv(index_csv)
        self.df = full[full["split"] == split].reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform
        self.labels = self.df[DISEASE_COLS].astype(np.float32).to_numpy()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, f"{row['image_id']}.jpg")).convert("RGB")
        if self.transform: img = self.transform(img)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return (img, y, str(row["patient_id"]),
                int(row.get("patient_sex", -1)),
                str(row.get("camera", "")),
                str(row.get("quality", "")),
                float(row["patient_age"]) if pd.notna(row.get("patient_age")) else -1.0)


print("Dataset + transforms defined.")

## Cell 3 — Build DINOv2 + 12-class head

DINOv2 from torch.hub returns just the backbone (CLS feature output). We wrap it in a small classifier that adds a `nn.Linear(1024, 12)` head.

In [ ]:
print("Loading DINOv2 ViT-L/14...")
# torch.hub validates the repo via GitHub API which can 504. Retry up to 3 times,
# then fall back to skip_validation=True (we trust facebookresearch/dinov2).
import time as _time
for attempt in range(3):
    try:
        backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14",
                                   verbose=False, trust_repo=True, skip_validation=True)
        break
    except Exception as e:
        print(f"  attempt {attempt+1} failed: {type(e).__name__}: {e}")
        if attempt == 2:
            raise
        _time.sleep(10)

print(f"DINOv2 loaded: {sum(p.numel() for p in backbone.parameters())/1e6:.1f}M params")

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, embed_dim=1024, num_classes=N_LABELS):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

model = DINOv2Classifier(backbone).to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
print(f"\nDINOv2 + 12-class head: {n_total/1e6:.1f}M trainable params (all unfrozen for fine-tuning)")


## Cell 4 — Loaders + per-label pos_weight + sanity batch

In [ ]:
BATCH_SIZE = 24  # same as RETFound FT
NUM_WORKERS = 2

train_ds = BRSETDataset(INDEX_CSV, "train", IMAGES_DIR, transform=train_tf())
val_ds   = BRSETDataset(INDEX_CSV, "val",   IMAGES_DIR, transform=eval_tf())
test_ds  = BRSETDataset(INDEX_CSV, "test",  IMAGES_DIR, transform=eval_tf())
print(f"train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

n_pos = train_ds.labels.sum(axis=0)
n_neg = len(train_ds) - n_pos
pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1), dtype=torch.float32).to(DEVICE)
print(f"\nPer-label pos_weight (truncated): {dict(zip(DISEASE_COLS[:4], [f'{w:.1f}' for w in pos_weight.cpu().tolist()[:4]]))}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Sanity batch
imgs, labels, *_ = next(iter(train_loader))
print(f"\nBatch shape: {imgs.shape}  labels: {labels.shape}")
with torch.no_grad():
    out = model(imgs.to(DEVICE))
print(f"Forward pass OK. Output shape: {out.shape}  (expected: [{BATCH_SIZE}, {N_LABELS}])")

## Cell 5 — Training loop (same recipe as RETFound FT)

AdamW lr=1e-4, wd=0.05, cosine + 2-epoch warmup, AMP, BCE + pos_weight, early stop patience 6, max 20 epochs.

In [ ]:
EPOCHS = 20
LR = 1e-5  # 10x lower than v1 — DINOv2 fine-tuning needs gentler LR
WD = 0.05
WARMUP_EPOCHS = 5  # longer warmup than v1
PATIENCE = 6

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    losses, all_logits, all_labels = [], [], []
    crit = nn.BCEWithLogitsLoss()
    for x, y, *_ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x); loss = crit(logits, y)
        losses.append(loss.item() * x.size(0))
        all_logits.append(logits.float().cpu()); all_labels.append(y.cpu())
    n = sum(t.size(0) for t in all_labels)
    avg_loss = sum(losses) / n
    logits = torch.cat(all_logits); labels = torch.cat(all_labels)
    probs = torch.sigmoid(logits).numpy()
    y_true = labels.numpy()
    aurocs = []
    for i in range(N_LABELS):
        y_bin = y_true[:, i]
        if 0 < y_bin.sum() < len(y_bin):
            aurocs.append(roc_auc_score(y_bin, probs[:, i]))
    return avg_loss, float(np.mean(aurocs)) if aurocs else float('nan')

def cosine_warmup_lr(epoch, total, warmup, base):
    if epoch < warmup: return base * (epoch + 1) / warmup
    p = (epoch - warmup) / max(1, total - warmup)
    return base * 0.5 * (1 + np.cos(np.pi * p))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_auroc = -1.0; best_epoch = -1; no_improve = 0
history = []

for epoch in range(EPOCHS):
    lr_now = cosine_warmup_lr(epoch, EPOCHS, WARMUP_EPOCHS, LR)
    for g in optimizer.param_groups: g["lr"] = lr_now
    model.train()
    t0 = time.time(); running = 0.0; n_seen = 0
    for x, y, *_ in train_loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x); loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item() * x.size(0); n_seen += x.size(0)
    train_loss = running / n_seen
    val_loss, val_auroc = evaluate(model, val_loader, DEVICE)
    dur = time.time() - t0
    history.append({"epoch": epoch, "lr": lr_now, "train_loss": train_loss,
                    "val_loss": val_loss, "val_macro_auroc": val_auroc, "seconds": dur})
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  lr={lr_now:.2e}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_macro_auroc={val_auroc:.4f}  ({dur:.0f}s)")
    if val_auroc > best_auroc:
        best_auroc = val_auroc; best_epoch = epoch; no_improve = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch,
                    "val_macro_auroc": val_auroc, "labels": DISEASE_COLS}, CKPT)
        print(f"  -> new best val_macro_auroc={val_auroc:.4f}, saved {CKPT}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}"); break

pd.DataFrame(history).to_csv(RESULTS / "training_history.csv", index=False)
print(f"\nBest val_macro_auroc={best_auroc:.4f} at epoch {best_epoch+1}")

## Cell 6 — Test eval with bootstrap CIs

In [ ]:
state = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state"]); model.eval()
print(f"Loaded checkpoint epoch {state['epoch']+1}  val_macro_auroc={state['val_macro_auroc']:.4f}")

@torch.no_grad()
def run_inference(model, loader, device):
    all_probs, all_y = [], []
    all_pid, all_sex, all_cam, all_qual, all_age = [], [], [], [], []
    for x, y, pid, sex, cam, qual, age in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_y.append(y.numpy())
        all_pid.extend(list(pid))
        all_sex.extend([int(s) for s in sex])
        all_cam.extend(list(cam)); all_qual.extend(list(qual))
        all_age.extend([float(a) for a in age])
    return (np.concatenate(all_probs), np.concatenate(all_y),
            np.array(all_pid), np.array(all_sex), np.array(all_cam),
            np.array(all_qual), np.array(all_age))

probs, y_true, pids, sex, cam, qual, age = run_inference(model, test_loader, DEVICE)
print(f"Test inference done. shape={probs.shape}")

def bootstrap_auroc(p, y, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y)
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if 0 < yi.sum() < len(yi):
            aurocs.append(roc_auc_score(yi, pi))
            auprcs.append(average_precision_score(yi, pi))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return dict(
        n=int(n), n_pos=int(y.sum()),
        auroc=float(np.mean(aurocs)) if aurocs else None,
        auroc_ci95=ci(aurocs) if aurocs else None,
        auprc=float(np.mean(auprcs)) if auprcs else None,
    )

per_label = {}
aurocs_all = []
print("\n=== Per-label test AUROC (DINOv2 fine-tuned on BRSET) ===")
for i, d in enumerate(DISEASE_COLS):
    r = bootstrap_auroc(probs[:, i], y_true[:, i])
    per_label[d] = r; aurocs_all.append(r["auroc"])
    print(f"  {d:30s}  n_pos={r['n_pos']:4d}  AUROC={r['auroc']:.4f}  CI={r['auroc_ci95']}")
macro_auroc = float(np.mean(aurocs_all))
print(f"\nMacro AUROC: {macro_auroc:.4f}")

## Cell 7 — Fairness audit (bootstrap CIs)

In [ ]:
MIN_N = 30; MIN_POS = 10

def audit_subgroup(p, y, mask, name):
    if mask.sum() < MIN_N: return None
    pi, yi = p[mask], y[mask]
    out = {"subgroup": name, "n": int(mask.sum()), "per_label": {}}
    for i, d in enumerate(DISEASE_COLS):
        if yi[:, i].sum() >= MIN_POS and yi[:, i].sum() < len(yi):
            out["per_label"][d] = bootstrap_auroc(pi[:, i], yi[:, i])
        else:
            out["per_label"][d] = None
    return out

audit = {}
audit["by_sex"] = [a for a in [
    audit_subgroup(probs, y_true, sex == 1, "sex=male"),
    audit_subgroup(probs, y_true, sex == 2, "sex=female"),
] if a]
audit["by_camera"] = [a for a in [
    audit_subgroup(probs, y_true, cam == c, f"camera={c}")
    for c in sorted(set(cam)) if c
] if a]
audit["by_quality"] = [a for a in [
    audit_subgroup(probs, y_true, qual == q, f"quality={q}")
    for q in sorted(set(qual)) if q
] if a]
known_age = age >= 0
audit["by_age_band"] = [a for a in [
    audit_subgroup(probs, y_true, known_age & (age >= lo) & (age < hi), f"age={name}")
    for (lo, hi, name) in [(0,40,"<40"),(40,60,"40-60"),(60,80,"60-80"),(80,200,"80+")]
] if a]

def find_gaps(rows, axis):
    print(f"\n=== Subgroup gaps where 95% CIs don't overlap ({axis}) ===")
    found = False
    for d in DISEASE_COLS:
        results = []
        for r in rows:
            v = r["per_label"].get(d)
            if v: results.append((r["subgroup"], v["auroc"], v["auroc_ci95"]))
        if len(results) < 2: continue
        for i in range(len(results)):
            for j in range(i+1, len(results)):
                a, b = results[i], results[j]
                if a[2][1] < b[2][0] or b[2][1] < a[2][0]:
                    found = True
                    print(f"  {d:30s}  {a[0]}: {a[1]:.3f} {a[2]}  vs  {b[0]}: {b[1]:.3f} {b[2]}")
    if not found: print("  (no statistically significant gaps)")

find_gaps(audit["by_sex"], "sex")
find_gaps(audit["by_camera"], "camera")
find_gaps(audit["by_quality"], "quality")
find_gaps(audit["by_age_band"], "age band")

## Cell 8 — Per-label calibration (ECE)

In [ ]:
def expected_calibration_error(probs, y_true, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; total = len(y_true)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (probs >= lo) & (probs < hi)
        if i == n_bins - 1: mask = (probs >= lo) & (probs <= hi)
        if mask.sum() == 0: continue
        bin_acc = float(y_true[mask].mean())
        bin_conf = float(probs[mask].mean())
        ece += (mask.sum() / total) * abs(bin_acc - bin_conf)
    return float(ece)

per_label_ece = {}
print("\n=== Per-label ECE ===")
for i, d in enumerate(DISEASE_COLS):
    e = expected_calibration_error(probs[:, i], y_true[:, i])
    per_label_ece[d] = e
    print(f"  {d:30s}  ECE={e:.4f}")
macro_ece = float(np.mean(list(per_label_ece.values())))
print(f"\nMacro ECE: {macro_ece:.4f}")

## Cell 9 — Persist + final comparison table

In [ ]:
results = {
    "method": "dinov2_full_finetune",
    "best_epoch": int(state["epoch"] + 1),
    "best_val_macro_auroc": float(state["val_macro_auroc"]),
    "test_n": int(len(y_true)),
    "test_macro_auroc": macro_auroc,
    "per_label": per_label,
    "fairness_audit": audit,
    "per_label_ece": per_label_ece,
    "macro_ece": macro_ece,
    "labels": DISEASE_COLS,
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved test_metrics.json")

print("\n=== FULL comparison table (all 5 conditions) ===")
print(f"  ResNet-50 supervised:         macro AUROC 0.9211   ECE (not measured)")
print(f"  RETFound linear probe:        macro AUROC 0.7842   macro ECE 0.3038")
print(f"  DINOv2 linear probe:          macro AUROC 0.8965   macro ECE 0.1562")
print(f"  RETFound fine-tuned:          macro AUROC 0.8962   macro ECE 0.0695")
print(f"  DINOv2 fine-tuned v1 (LR=1e-4): macro AUROC 0.7293   macro ECE 0.3624")
print(f"  DINOv2 fine-tuned v2 (LR=1e-5): macro AUROC {macro_auroc:.4f}   macro ECE {macro_ece:.4f}")

import shutil
zip_path = WORK / "dinov2_finetune_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"\nWrote {zip_path}")